<a href="https://colab.research.google.com/github/shishirkumar12/AIMl/blob/main/Chat_with_Your_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 1: Environment Setup (Zaroori Tools Install Karna)
Notebook ki sabse pehli cell mein humne server par saare required Python packages install kiye:

In [1]:
# Cell [1]: Saare packages install karna
!pip install -q langchain langchain-community faiss-cpu sentence-transformers pypdf langchain-text-splitters langchain-huggingface langchain-google-genai gradio streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Har Ek Package Ka Kaam Kya Hai?
pypdf: Yeh aapke computer ke bhasha (Python) mein PDF files ko open karne aur uske andar chhupe text ko dhoodh kar nikalne ka kaam karta hai.

langchain-text-splitters: Iska kaam hai poori PDF ke hazaron lines ke text ko chhote-chhote, readable parat (chunks) mein todna.

sentence-transformers & langchain-huggingface: Yeh models insani bhasha (text) ko samajhte hain aur use mathematical numbers (vectors/embeddings) mein badalte hain.

faiss-cpu: Facebook ka Vector Database hai. Iska kaam un saare mathematical numbers (vectors) ko memory mein hold karna aur unme se fast searching karna hai.

langchain-google-genai: Iska kaam humare code ko Google ke Gemini AI (LLM) server se safe aur securely connect karna hai.

streamlit: Ek aasan python framework jo humein HTML/CSS/JS sikhe bina ek professional aur responsive web UI (frontend website) bana kar deta hai.

Phase 2: Core Connection Setup (AI credentials)
Yahan humne aapki safe rakhi hui Gemini API key ko load kiya aur main Language Model (LLM) ko taiyar kiya.


In [3]:
# Cell [17]: LLM Initialisation
import os
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
# Colab ke Secrets panel (🔑 icon) se safe rakhi hui API key ko env variable me set kiya
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [5]:
# Gemini-2.5-flash model ko select kiya aur temperature=0 rakha.
# temperature=0 rakhne se AI bilkul accurate rehta hai, jhoot nahi bolta (hallucinations stop ho jate hain).
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

Phase 3: The Frontend UI & Dynamic Logic (app.py)
Humne %%writefile app.py command ka use kiya jo pure notebook ke code ko ek alag script file (app.py) mein daal deta hai, jise baad mein Streamlit engine chala pata hai. Is single file ke andar poore project ki dynamic workflow chhupi hai:

In [6]:
%%writefile app.py
import os
import streamlit as st
from google.colab import userdata
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI

# =========================================================================
# 1. PAGE SETUP & TITLE
# =========================================================================
st.set_page_config(page_title="Dynamic AI PDF Chatbot", page_icon="🤖", layout="wide")
st.title("🤖 Chat with Your PDF")
st.caption("Upload any PDF file and start asking questions instantly!")

# =========================================================================
# 2. GEMINI ENGINE & EMBEDDING INITS
# =========================================================================
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# @st.cache_resource lagaya taaki embedding model har user click par reload na ho (Memory bachegi)
@st.cache_resource
def get_embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embeddings = get_embedding_model()

# =========================================================================
# 3. SIDEBAR: LIVE DYNAMIC PDF UPLOADER ENGINE
# =========================================================================
st.sidebar.header("📁 Document Upload")
uploaded_file = st.sidebar.file_uploader("Upload your PDF file", type=["pdf"])

db = None  # Shuruat me database empty (khali) rahega

if uploaded_file is not None:
    # st.session_state ka use kiya taaki website reload hone par vector database baar-baar scratch se na bane
    if "vector_db" not in st.session_state:
        with st.sidebar.spinner("Processing PDF... Please wait."):

            # --- STEP A: Text aur Page Number Nikalna ---
            reader = PdfReader(uploaded_file)
            docs_list = []
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    # page metadata me page index i rakh rhe hain taaki baad me source track ho sake
                    docs_list.append(Document(page_content=text, metadata={"page": i}))

            # --- STEP B: Text Chunks Splitter ---
            # 1000 size ke chunks banenge jisme 200 overlap hoga (taki context kategi nahi)
            text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
            final_chunks = text_splitter.split_documents(docs_list)

            # --- STEP C: Live Vector DB Generation ---
            # Is line me pure chunks ko embedding model mathematical numbers me convert karke FAISS DB me load kar deta hai
            st.session_state.vector_db = FAISS.from_documents(final_chunks, embeddings)
            st.sidebar.success(f"✅ PDF Processed! {len(final_chunks)} chunks created.")

    db = st.session_state.vector_db
else:
    # Agar user file remove kar de, toh session state se purana database delete ho jaye
    if "vector_db" in st.session_state:
        del st.session_state.vector_db

# =========================================================================
# 4. MAIN WINDOW: THE CORE RAG CHAT ENGINE
# =========================================================================
if db is not None:
    question = st.text_input("Ask a question about your PDF:", placeholder="e.g., What is the main summary?")

    if question:
        with st.spinner("Analyzing document..."):

            # --- STEP 1: Similarity Search ---
            # FAISS vector database aapke question se match karte hue top 3 chunks filter karke laayega
            results = db.similarity_search(question, k=3)
            context = "\n\n".join([doc.page_content for doc in results])

            # --- STEP 2: Prompt Engineering (Strict Control) ---
            # AI ko ek boundry ke andar lock karne ke liye prompt diya gaya hai
            prompt = f"""
            Answer only from the context.
            Context:
            {context}
            Question:
            {question}
            If the answer is not found, say: I couldn't find the answer in the uploaded PDF.
            """

            # --- STEP 3: Invoke LLM ---
            response = llm.invoke(prompt)

            # --- STEP 4: Source Extract Logic ---
            # Set use kiya hai taaki duplicate page numbers remove ho jayein
            sources = set()
            for doc in results:
                page = doc.metadata.get("page")
                if isinstance(page, int):
                    sources.add(f"Page {page + 1}") # Python index 0 se ginta hai, user ke liye +1 kiya

            # --- STEP 5: Beautiful Output Display ---
            st.write("### 📝 Answer")
            st.write(response.content) # Main output from Gemini

            st.write("### 📍 Sources")
            for source in sorted(sources):
                st.info(source) # Blue alert box me safe information page number dikhane ke liye
else:
    # Agar koi PDF upload nahi hai toh screen par guidance message dikhega
    st.info("👈 Please upload a PDF file from the sidebar to start chatting.")

Writing app.py


Phase 4: Secure Deployment (Pinggy Tunneling)
Kyunki Google Colab ek private machine hai jahan chalne wale internal system local network (localhost:8501) par block hote hain, humne Pinggy Secure Tunnel ke zariye use ek public secure live connection par badal diya.

In [ ]:
# Cell [33]: Final Connection Run
import subprocess
import urllib.request

# 1. Streamlit app.py ko background process me chupchaap chalaya port 8501 par
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])

# 2. Python se Colab runtime ke computer ka actual public IP address dhoodha gaya
try:
    ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
    print("==================================================")
    print(f"YOUR TUNNEL PASSWORD (IP): {ip}")  # Output: 35.237.253.120
    print("==================================================")
except Exception as e:
    print("Could not fetch IP automatically.")

# 3. SSH command chalakar Pinggy ke network ko bypass (-T) aur trusted checking (-o) flag ke sath link kiye
!ssh -p 443 -T -o StrictHostKeyChecking=no -R0:localhost:8501 qr@a.pinggy.io

YOUR TUNNEL PASSWORD (IP): 34.75.29.221
Allocated port 1 for remote forward to localhost:8501
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
http://nihqr-34-75-29-221.run.pinggy-free.link
http://psjar-34-75-29-221.free.pinggy.net
https://nihqr-34-75-29-221.run.pinggy-free.link
https://psjar-34-75-29-221.free.pinggy.net
RB: 106883, SB: 3200026, TC: 19, AC: 1               

Final Project Workflow Summary
Jab aap generated Pinggy link ko open karte hain toh yeh pure technical chain sequential order mein kaam karti hai:

User Visit: User ne live website url open kiya, use ek saaf UI dikha.

Dynamic Reading: User ne koi bhi random PDF file upload ki, Streamlit ke backend process (PdfReader) ne turant use memory mein chunks mein toda.

Smart Conversion: HuggingFace Embeddings ne un chunks ko vectors mein badal kar FAISS in-memory setup kar diya.

Semantic Search: Jab user ne question pucha, FAISS ne context match karke top 3 blocks uthaye.

Generative Processing: Gemini AI ne un contexts ko strict boundary mein read karke accurate answer ready kiya aur UI par display ho gaya.

Aapka poora project ab bina kisi error ke fully absolute independent tarike se chalne ke liye ready hai!